In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import os
import warnings
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

# -----------------------------
# Setup
# -----------------------------
os.makedirs("../reports/figures", exist_ok=True)

# -----------------------------
# Load Data
# -----------------------------
DATA_PATH = "../data/processed/network_logs_preprocessed.csv"
df = pd.read_csv(DATA_PATH)

if "timestamp" in df.columns:
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp")

# -----------------------------
# Train/Test Split
# -----------------------------
if 'label' in df.columns:
    train_df, test_df = train_test_split(
        df, test_size=0.2, random_state=42, stratify=df['label']
    )
else:
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# -----------------------------
# Feature Selection
# -----------------------------
core_features = ['dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sport', 'dsport']
proto_features = [
    'proto_tcp', 'proto_udp', 'proto_icmp', 'proto_arp',
    'proto_esp', 'proto_gre', 'proto_ipcomp', 'proto_ipv6', 'proto_sctp'
]

feature_cols = [col for col in (core_features + proto_features) if col in df.columns]

X_test = test_df[feature_cols].copy()

# -----------------------------
# Load Scaler (FIXED)
# -----------------------------
scaler = pickle.load(open("../models/scaler.pkl", "rb"))
X_test_scaled = scaler.transform(X_test)

# -----------------------------
# Load Model
# -----------------------------
model = pickle.load(open("../models/isolation_forest.pkl", "rb"))

# -----------------------------
# Predictions
# -----------------------------
y_pred_raw = model.predict(X_test_scaled)
y_pred = np.where(y_pred_raw == -1, 1, 0)

# Anomaly score (IMPORTANT)
scores = model.decision_function(X_test_scaled)

test_df = test_df.copy()
test_df["anomaly"] = y_pred
test_df["anomaly_score"] = scores

print(f"Detected {y_pred.sum():,} anomalies")

# Rank anomalies (LOW score = more anomalous)
test_df = test_df.sort_values("anomaly_score")

# -----------------------------
# Save Top Anomalies
# -----------------------------
top_anomalies = test_df[test_df["anomaly"] == 1].head(100)
top_anomalies.to_csv("../reports/top_100_anomalies.csv", index=False)


# -----------------------------
# 1. GLOBAL FEATURE IMPORTANCE
# -----------------------------
if hasattr(model, "feature_importances_"):
    importances = model.feature_importances_
    feat_imp = pd.Series(importances, index=feature_cols).sort_values(ascending=False)

    print("\nGlobal Feature Importance:\n", feat_imp.head(10))

    plt.figure()
    feat_imp.head(10).plot(kind="bar")
    plt.title("Global Feature Importance")
    plt.tight_layout()
    plt.savefig("../reports/figures/global_feature_importance.png")
    plt.show()

In [ ]:

# -----------------------------
# 2. SHAP EXPLAINER
# -----------------------------
print("\nInitializing SHAP explainer...")

explainer = shap.TreeExplainer(model)

sample_size = 2000
X_sample = X_test_scaled[:sample_size]

shap_values = explainer.shap_values(X_sample)

# -----------------------------
# 3. SHAP SUMMARY PLOT
# -----------------------------
print("Generating SHAP summary plot...")

shap.summary_plot(shap_values, X_sample, feature_names=feature_cols, show=False)
plt.tight_layout()
plt.savefig("../reports/figures/shap_summary.png")
plt.show()

In [ ]:

# -----------------------------
# 4. SHAP BAR PLOT
# -----------------------------
shap.summary_plot(
    shap_values,
    X_sample,
    feature_names=feature_cols,
    plot_type="bar",
    show=False
)
plt.tight_layout()
plt.savefig("../reports/figures/shap_bar.png")
plt.show()

In [ ]:

# -----------------------------
# 5. LOCAL EXPLANATION (BEST VERSION)
# -----------------------------
print("\nExplaining a specific anomaly...")

anomalies = test_df[test_df["anomaly"] == 1]

if not anomalies.empty:
    test_df = test_df.reset_index(drop=True)
    anomalies = test_df[test_df["anomaly"] == 1]

    loc = anomalies.index[0]

    sample = X_test_scaled[loc]

    shap_val_single = explainer.shap_values(sample)

    if isinstance(shap_val_single, list):
        shap_val_single = shap_val_single[0]

    print(f"Explaining anomaly at position {loc}")


    base_value = explainer.expected_value

    if isinstance(base_value, (list, np.ndarray)):
        base_value = base_value[0]

    
    shap.plots.waterfall(
        shap.Explanation(
            values=shap_val_single,
            base_values=base_value,
            data=sample,
            feature_names=feature_cols
        ),
        show=False
    )

    plt.savefig("../reports/figures/shap_waterfall.png")
    plt.show()


In [ ]:

# -----------------------------
# 6. ANOMALY SCORE DISTRIBUTION
# -----------------------------
plt.figure()
sns.histplot(test_df["anomaly_score"], bins=50)
plt.title("Anomaly Score Distribution")
plt.savefig("../reports/figures/anomaly_score_distribution.png")
plt.show()


In [ ]:


# -----------------------------
# 7. OPERATIONAL INSIGHT (AUTO TEXT)
# -----------------------------
print("\nGenerating operational insight...")

if not anomalies.empty:
    anomaly_features = anomalies[feature_cols].iloc[0]
    normal_means = test_df[test_df["anomaly"] == 0][feature_cols].mean()

    diff = (anomaly_features - normal_means).sort_values(ascending=False)

    print("\nTop contributing features:")
    print(diff.head(5))

    explanation = "This session was flagged due to: "
    explanation += ", ".join([f"{feat} unusually high" for feat in diff.head(5).index])

    print("\nAuto-generated explanation:")
    print(explanation)

# -----------------------------
# Save Full Results
# -----------------------------
test_df.to_csv("../reports/full_results_with_explanations.csv", index=False)
